# Học deep learning với dữ liệu chữ số viết tay MNIST

Notebook này dạy từng bước: dữ liệu ảnh dưới dạng mảng số, mô hình baseline, mạng nơ-ron nhiều lớp (MLP) bằng scikit-learn và Keras, mạng tích chập (CNN) bằng Keras, và cách phân tích lỗi của mô hình phân loại ảnh.

Dữ liệu: 70.000 ảnh xám kích thước 28 x 28 pixel, mỗi ảnh là một chữ số từ 0 đến 9. Có 60.000 ảnh train và 10.000 ảnh test, đã chia sẵn. Mỗi pixel có giá trị từ 0 (đen) đến 255 (trắng). Nguồn: MNIST của LeCun, Cortes và Burges.

Chuẩn bị:
1. Đặt file data9.npz cùng thư mục với notebook. File .npz là định dạng nén của numpy, chứa 4 mảng: X_train, y_train, X_test, y_test.
2. Cài thư viện nếu chưa có: pip install numpy pandas matplotlib scikit-learn tensorflow
3. Các bước dùng Keras (từ bước 10) cần tensorflow. Trên CPU, mỗi bước huấn luyện mất khoảng 1 đến 3 phút. Kết quả của mạng nơ-ron có thể lệch vài phần nghìn so với kết quả mong đợi vì khác máy và khác phiên bản thư viện.

Cách làm: đọc phần giải thích, viết code vào ô trống, chạy bằng Shift + Enter, rồi so sánh với kết quả mong đợi. Nếu kết quả khác, đọc lại đề bài và thử lại trước khi xem đáp án.

## Bước 0: Import thư viện

Mục tiêu: nạp các thư viện dùng trong toàn bộ notebook. Ô này đã viết sẵn, bạn chỉ cần chạy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

## Bước 1: Đọc dữ liệu

Mục tiêu: mở file .npz và kiểm tra kích thước các mảng.

Kiến thức: np.load trả về một đối tượng giống dict. data["X_train"] lấy ra mảng tương ứng. X_train có 3 chiều: (số ảnh, chiều cao, chiều rộng).

Kết quả mong đợi: X_train (60000, 28, 28), y_train (60000,), X_test (10000, 28, 28), y_test (10000,). Kiểu dữ liệu uint8, giá trị pixel từ 0 đến 255.

In [ ]:
# Viết code ở đây:
# 1. data = np.load('data9.npz')
# 2. Gán X_train, y_train, X_test, y_test từ data
# 3. In shape của 4 mảng, dtype của X_train, min và max của X_train


## Bước 2: Xem ảnh

Mục tiêu: nhìn tận mắt dữ liệu trước khi huấn luyện.

Kiến thức: plt.imshow(ảnh, cmap="gray") vẽ một mảng 2 chiều thành ảnh xám. Một ảnh chỉ là một bảng 28 x 28 con số.

Kết quả mong đợi: 10 ảnh đầu là các chữ số 5, 0, 4, 1, 9, 2, 1, 3, 1, 4.

In [ ]:
# Viết code ở đây:
# 1. Tạo fig, axes = plt.subplots(2, 5, figsize=(10, 4))
# 2. Với i từ 0 đến 9: vẽ X_train[i] lên axes.flat[i], đặt tiêu đề là y_train[i], tắt trục
# 3. In mảng X_train[0][5:15, 5:15] để thấy giá trị pixel


## Bước 3: Phân phối nhãn

Mục tiêu: kiểm tra các chữ số có số lượng gần bằng nhau không.

Kiến thức: np.bincount đếm số lần xuất hiện của mỗi số nguyên không âm.

Kết quả mong đợi: mỗi chữ số có khoảng 5.400 đến 6.700 ảnh trong tập train. Chữ số 1 nhiều nhất trong tập test (1135 ảnh), nên baseline đoán toàn số 1 đạt khoảng 0,1135.

In [ ]:
# Viết code ở đây:
# 1. In np.bincount(y_train) và np.bincount(y_test)
# 2. In tỷ lệ của chữ số xuất hiện nhiều nhất trong tập test


## Bước 4: Duỗi ảnh và chuẩn hóa

Mục tiêu: chuyển mỗi ảnh thành một vector 784 số trong khoảng 0 đến 1.

Kiến thức: mô hình của scikit-learn cần X 2 chiều (số mẫu, số feature). reshape(len, -1) duỗi 28 x 28 thành 784 cột. Chia cho 255 đưa giá trị về khoảng 0 đến 1, giúp mạng nơ-ron học ổn định hơn. Cách duỗi này làm mất thông tin pixel nào nằm cạnh pixel nào, CNN ở bước 12 sẽ giữ lại thông tin đó.

Kết quả mong đợi: X_train_flat có kích thước (60000, 784), giá trị lớn nhất bằng 1,0.

In [ ]:
# Viết code ở đây:
# 1. X_train_flat = X_train.reshape(len(X_train), -1) / 255.0
# 2. Làm tương tự cho X_test_flat
# 3. In shape và max của X_train_flat


## Bước 5: Baseline và logistic regression

Mục tiêu: có mốc so sánh trước khi dùng mạng nơ-ron.

Kiến thức: DummyClassifier luôn đoán chữ số xuất hiện nhiều nhất. Logistic regression đa lớp học một bộ trọng số 784 số cho mỗi chữ số. Để chạy nhanh, ta chỉ huấn luyện logistic trên 10.000 ảnh đầu. Bước này mất khoảng 30 giây.

Kết quả mong đợi: baseline khoảng 0,1135. Logistic regression trên 10.000 ảnh đạt khoảng 0,905.

In [ ]:
# Viết code ở đây:
# 1. Fit DummyClassifier(strategy='most_frequent') trên X_train_flat, y_train, in accuracy trên tập test
# 2. Fit LogisticRegression(max_iter=200) trên 10000 ảnh đầu
# 3. In accuracy trên toàn bộ tập test


## Bước 6: Trọng số của logistic regression

Mục tiêu: hiểu logistic regression "nhìn" gì trong ảnh.

Kiến thức: coef_ có kích thước (10, 784). Reshape mỗi dòng về 28 x 28 và vẽ ra: vùng đỏ là pixel sáng làm tăng xác suất của chữ số đó, vùng xanh làm giảm.

Kết quả mong đợi: mỗi hình trông như một bản mờ của chữ số tương ứng. Ví dụ hình của số 0 có một vòng tròn đỏ với vùng xanh ở giữa.

In [ ]:
# Viết code ở đây:
# 1. Tạo fig, axes = plt.subplots(2, 5, figsize=(10, 4))
# 2. Với mỗi chữ số i: vẽ logreg.coef_[i].reshape(28, 28) bằng cmap='RdBu_r', đặt tiêu đề i, tắt trục


## Bước 7: MLP bằng scikit-learn

Mục tiêu: huấn luyện mạng nơ-ron đầu tiên.

Kiến thức: MLP (multi-layer perceptron) gồm lớp vào (784 nơ-ron), các lớp ẩn và lớp ra (10 nơ-ron). Mỗi nơ-ron ẩn tính tổng có trọng số của đầu vào rồi áp dụng hàm kích hoạt ReLU, max(0, x). hidden_layer_sizes=(128,) là một lớp ẩn 128 nơ-ron. early_stopping=True giữ 10% tập train làm validation và dừng khi điểm validation không cải thiện. Bước này mất khoảng 1 đến 2 phút.

Kết quả mong đợi: accuracy khoảng 0,978, chạy hết 30 epoch. Có thể xuất hiện ConvergenceWarning vì mô hình chưa hội tụ hoàn toàn sau 30 epoch, bạn có thể bỏ qua.

In [ ]:
# Viết code ở đây:
# 1. Tạo MLPClassifier(hidden_layer_sizes=(128,), max_iter=30, early_stopping=True, random_state=42)
# 2. Fit trên X_train_flat, y_train
# 3. In accuracy trên tập test và số epoch đã chạy (mlp.n_iter_)


## Bước 8: Đường loss và số tham số

Mục tiêu: xem mạng học thế nào qua từng epoch và đếm số tham số.

Kiến thức: một epoch là một lần đi qua toàn bộ dữ liệu train. loss_curve_ lưu loss trên tập train sau mỗi epoch. validation_scores_ lưu accuracy trên phần validation. Số tham số của một lớp dense = số đầu vào x số nơ-ron + số nơ-ron (bias).

Kết quả mong đợi: loss giảm nhanh trong vài epoch đầu rồi giảm chậm dần. Validation accuracy tăng lên khoảng 0,98 rồi gần như đi ngang. MLP có 101.770 tham số: 784 x 128 + 128 ở lớp ẩn và 128 x 10 + 10 ở lớp ra.

In [ ]:
# Viết code ở đây:
# 1. Vẽ mlp.loss_curve_ và mlp.validation_scores_ trên 2 hình cạnh nhau
# 2. Tính số tham số: tổng của w.size cho w trong mlp.coefs_ cộng tổng b.size cho b trong mlp.intercepts_


## Bước 9: Confusion matrix và ảnh bị đoán sai

Mục tiêu: tìm cặp chữ số mà mô hình hay nhầm.

Kiến thức: với 10 lớp, confusion matrix có 10 x 10 ô. Để tìm cặp hay nhầm nhất, đặt đường chéo bằng 0 rồi tìm ô lớn nhất. Xem tận mắt các ảnh bị sai thường cho thấy một số ảnh khó ngay cả với người.

Kết quả mong đợi: khoảng 216 ảnh bị sai. Các cặp hay nhầm nhất là thật 4 đoán 9, thật 7 đoán 2 và thật 2 đoán 7 (mỗi cặp khoảng 9 đến 10 ảnh).

Câu hỏi: trong các ảnh bị sai, có ảnh nào bạn cũng không chắc là số gì không?

In [ ]:
# Viết code ở đây:
# 1. pred_mlp = mlp.predict(X_test_flat)
# 2. Vẽ ConfusionMatrixDisplay.from_predictions(y_test, pred_mlp)
# 3. cm = confusion_matrix(...), np.fill_diagonal(cm, 0), in 3 cặp (thật, dự đoán) có số lỗi lớn nhất
# 4. Vẽ 10 ảnh đầu tiên bị đoán sai, tiêu đề dạng 'thật 4, đoán 9'


## Bước 10: MLP bằng Keras

Mục tiêu: xây lại MLP bằng Keras để làm quen với thư viện deep learning.

Kiến thức: Keras xây mạng bằng Sequential gồm các lớp xếp chồng. Dense(128, activation="relu") là lớp ẩn. Dropout(0.2) tắt ngẫu nhiên 20% nơ-ron mỗi bước để giảm overfitting. Lớp cuối Dense(10, activation="softmax") cho 10 xác suất cộng lại bằng 1. compile chọn optimizer (adam), hàm loss (sparse_categorical_crossentropy, dùng khi nhãn là số nguyên 0 đến 9) và chỉ số theo dõi. set_random_seed giúp kết quả gần như lặp lại được.

Kết quả mong đợi: summary cho thấy 101.770 tham số, giống MLP của sklearn. Sau 10 epoch, accuracy trên tập test khoảng 0,977.

In [ ]:
# Viết code ở đây:
# 1. Đặt os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' để ẩn thông báo kỹ thuật, rồi import tensorflow as tf, from tensorflow import keras; gọi keras.utils.set_random_seed(42)
# 2. Xây Sequential: keras.Input(shape=(784,)), Dense(128, relu), Dropout(0.2), Dense(10, softmax)
# 3. compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# 4. model_summary: gọi mlp_k.summary()
# 5. fit với epochs=10, batch_size=128, validation_split=0.1, verbose=2, lưu vào hist_mlp
# 6. In accuracy trên tập test bằng evaluate


## Bước 11: Đường học của Keras

Mục tiêu: phát hiện overfitting bằng cách so sánh train và validation qua các epoch.

Kiến thức: hist.history là dict chứa loss, accuracy, val_loss và val_accuracy cho mỗi epoch. Nếu loss train tiếp tục giảm mà val_loss đi ngang hoặc tăng, mô hình bắt đầu overfit.

Kết quả mong đợi: accuracy train tăng từ khoảng 0,88 lên khoảng 0,98. val_accuracy epoch cuối khoảng 0,980. val_loss vẫn giảm đến epoch 10, nên chưa thấy overfitting rõ.

In [ ]:
# Viết code ở đây:
# 1. Vẽ accuracy và val_accuracy trên một hình, loss và val_loss trên hình bên cạnh
# 2. In val_accuracy của epoch cuối (làm tròn 4 chữ số)


## Bước 12: CNN

Mục tiêu: xây mạng tích chập, giữ được cấu trúc không gian của ảnh.

Kiến thức: Conv2D(32, kernel_size=3) trượt 32 bộ lọc 3 x 3 trên ảnh. Mỗi bộ lọc học phát hiện một mẫu nhỏ như nét ngang, nét cong. MaxPooling2D(2) lấy giá trị lớn nhất trong mỗi ô 2 x 2, làm ảnh nhỏ đi một nửa. Flatten duỗi kết quả thành vector trước lớp Dense cuối. CNN cần đầu vào 4 chiều: (số ảnh, cao, rộng, số kênh màu), ảnh xám có 1 kênh. Bước này mất khoảng 2 đến 4 phút trên CPU.

Kết quả mong đợi: X_train_img có kích thước (60000, 28, 28, 1). CNN chỉ có 34.826 tham số. Sau 5 epoch, accuracy trên tập test khoảng 0,988.

In [ ]:
# Viết code ở đây:
# 1. X_train_img = X_train[..., np.newaxis] / 255.0, tương tự X_test_img; in shape
# 2. Gọi keras.utils.set_random_seed(42)
# 3. Sequential: Input((28, 28, 1)), Conv2D(32, 3, relu), MaxPooling2D(2), Conv2D(64, 3, relu), MaxPooling2D(2), Flatten(), Dropout(0.5), Dense(10, softmax)
# 4. compile như bước 10, gọi summary()
# 5. fit với epochs=5, batch_size=128, validation_split=0.1, verbose=2, lưu vào hist_cnn
# 6. In accuracy trên tập test


## Bước 13: Bộ lọc và feature map của CNN

Mục tiêu: xem lớp tích chập đầu tiên "thấy" gì trong một ảnh.

Kiến thức: feature map là kết quả khi một bộ lọc trượt qua ảnh. Có thể tạo một mô hình con trả về đầu ra của lớp Conv2D đầu tiên bằng keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output).

Kết quả mong đợi: 16 feature map đầu tiên của ảnh số 7. Một số map sáng ở nét ngang phía trên, một số sáng ở nét chéo, một số sáng ở viền.

In [ ]:
# Viết code ở đây:
# 1. feat_model = keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)
# 2. maps = feat_model.predict(X_test_img[:1], verbose=0); in maps.shape
# 3. Vẽ ảnh gốc X_test[0], rồi 16 feature map đầu tiên (maps[0, :, :, i]) trên lưới 2 x 8


## Bước 14: Bảng so sánh các mô hình

Mục tiêu: so sánh accuracy, số ảnh sai và số tham số của mọi mô hình.

Kiến thức: model.count_params() trả về số tham số của mô hình Keras. Số ảnh sai = 10000 x (1 - accuracy).

Kết quả mong đợi: CNN đứng đầu với accuracy khoảng 0,988 và khoảng 124 ảnh sai, dùng 34.826 tham số. MLP sklearn khoảng 216 ảnh sai và MLP Keras khoảng 228 ảnh sai, cả hai dùng 101.770 tham số. Logistic sai khoảng 950 ảnh.

Câu hỏi: CNN có ít tham số hơn MLP Keras mà vẫn chính xác hơn. Vì sao? Gợi ý: một bộ lọc 3 x 3 được dùng lại ở mọi vị trí trong ảnh.

In [ ]:
# Viết code ở đây:
# 1. Tính accuracy trên tập test cho: dummy, logreg, mlp (sklearn), mlp_k, cnn
# 2. Với mô hình Keras, dùng np.argmax(model.predict(X, verbose=0), axis=1) để lấy nhãn
# 3. Tạo DataFrame với cột: Mô hình, accuracy, số ảnh sai, số tham số; in ra


## Bước 15: Tự thử nghiệm

Không có code mẫu ở bước này. Hãy tự chọn ít nhất hai việc sau:

1. Thêm keras.layers.BatchNormalization() sau mỗi lớp Conv2D và huấn luyện lại CNN. Accuracy và thời gian huấn luyện thay đổi thế nào?
2. Huấn luyện MLP Keras 30 epoch thay vì 10, rồi vẽ đường học. Bạn có thấy overfitting không? Thử thêm keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True).
3. Tự vẽ một chữ số bằng Paint (nền đen, nét trắng, 28 x 28 pixel), đọc bằng plt.imread và cho CNN dự đoán. Mô hình có đoán đúng không?
4. Viết ba câu tiếng Anh so sánh MLP và CNN dựa trên bảng ở bước 14. Ví dụ: "The CNN reduced the number of misclassified test images from ... to ... while using fewer parameters."
